In [1]:
import os
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

# If you already set excel_filepath in an earlier cell, this keeps it.
# Otherwise it falls back to the repo copy of the metadata file.
excel_filepath = globals().get(
    "excel_filepath",
    r"/home/frederik/github_projects/SNPster/data pipeline/reporting_module/data/pgs_all_metadata.xlsx"
)


def _first_existing(series_or_df, candidates, required=True, default=None):
    cols = list(series_or_df.columns)

    def _norm(name):
        return " ".join(str(name).strip().lower().split())

    normalized_to_original = {_norm(col): col for col in cols}

    for candidate in candidates:
        if candidate in cols:
            return series_or_df[candidate]

        candidate_norm = _norm(candidate)
        if candidate_norm in normalized_to_original:
            return series_or_df[normalized_to_original[candidate_norm]]

    if required:
        raise KeyError(f"Missing required column. Expected one of: {candidates}")
    return pd.Series([default] * len(series_or_df), index=series_or_df.index)


def _to_nullable_int(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64")


def _to_bool(series):
    mapping = {
        "true": True,
        "false": False,
        "yes": True,
        "no": False,
        "y": True,
        "n": False,
        "1": True,
        "0": False,
    }
    normalized = series.astype(str).str.strip().str.lower()
    return normalized.map(mapping).where(~series.isna(), None)


def _parse_effect_with_ci(series):
    """
    Parse values like:
      1.55 [1.52,1.58]
      1.55 (1.52-1.58)
      1.55
    Returns three numeric series: point_estimate, ci_lower, ci_upper.
    """
    s = series.astype(str)

    # First numeric token is the point estimate.
    point = pd.to_numeric(
        s.str.extract(r"([+-]?(?:\d+(?:\.\d+)?|\.\d+))", expand=False),
        errors="coerce",
    )

    # Capture lower/upper inside [] or () using either comma or dash separator.
    ci_match = s.str.extract(
        r"[\[\(]\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+))\s*[,\-]\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+))\s*[\]\)]",
        expand=True,
    )
    ci_lower = pd.to_numeric(ci_match[0], errors="coerce")
    ci_upper = pd.to_numeric(ci_match[1], errors="coerce")

    return point, ci_lower, ci_upper


def _parse_point_estimate(series):
    """Parse the leading numeric point estimate from strings with optional CI text."""
    point, _, _ = _parse_effect_with_ci(series)
    return point


def _split_pgs_ids(value):
    """
    Split one or many PGS IDs from metadata cells like:
      PGS000001
      PGS000001, PGS000004, PGS000007
      PGS000001;PGS000004
    """
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    parts = [p.strip() for p in text.replace(";", ",").split(",")]
    return [p for p in parts if p]


def _clean_df_for_sql(df):
    cleaned = df.copy()
    cleaned = cleaned.where(pd.notna(cleaned), None)
    return cleaned


def _to_db_scalar(value):
    """Convert pandas/numpy scalar types to psycopg2-compatible Python scalars."""
    if value is None:
        return None

    # Handle pandas missing sentinels (pd.NA, NaN, NaT) as SQL NULL.
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    if isinstance(value, (pd.Timestamp, np.datetime64)):
        ts = pd.Timestamp(value)
        return ts.to_pydatetime()
    if isinstance(value, np.generic):
        return value.item()
    return value


def _get_table_columns(cursor, schema, table):
    cursor.execute(
        """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position
        """,
        (schema, table),
    )
    return [row[0] for row in cursor.fetchall()]


def _bulk_insert(cursor, schema, table, dataframe):
    if dataframe.empty:
        print(f"Skipping {schema}.{table}: no rows")
        return

    table_columns = _get_table_columns(cursor, schema, table)
    columns = [c for c in dataframe.columns if c in table_columns]
    skipped = [c for c in dataframe.columns if c not in table_columns]

    if skipped:
        print(f"Skipping non-existent columns for {schema}.{table}: {skipped}")
    if not columns:
        raise ValueError(f"No matching dataframe columns for {schema}.{table}")

    tuples = [
        tuple(_to_db_scalar(v) for v in row)
        for row in dataframe[columns].itertuples(index=False, name=None)
    ]
    query = f"INSERT INTO {schema}.{table} ({', '.join(columns)}) VALUES %s"
    execute_values(cursor, query, tuples, page_size=1000)
    print(f"Inserted {len(tuples)} rows into {schema}.{table}")


# -----------------------------
# 1) Read source sheets
# -----------------------------
pgscatalog_raw = pd.read_excel(excel_filepath, sheet_name="Scores")
publications_raw = pd.read_excel(excel_filepath, sheet_name="Publications")
ontology_raw = pd.read_excel(excel_filepath, sheet_name="EFO Traits")
performance_raw = pd.read_excel(excel_filepath, sheet_name="Performance Metrics")
score_dev_raw = pd.read_excel(excel_filepath, sheet_name="Score Development Samples")
eval_sets_raw = pd.read_excel(excel_filepath, sheet_name="Evaluation Sample Sets")


# -----------------------------
# 2) Map to DB schema columns
# -----------------------------
pgscatalog_data = pd.DataFrame({
    "pgs_id": _first_existing(pgscatalog_raw, ["Polygenic Score (PGS) ID"]),
    "pgs_name": _first_existing(pgscatalog_raw, ["PGS Name"], required=False),
    "reported_trait": _first_existing(pgscatalog_raw, ["Reported Trait"], required=False),
    "mapped_trait_efo_label": _first_existing(pgscatalog_raw, ["Mapped Trait(s) (EFO label)"], required=False),
    "efo_id": _first_existing(pgscatalog_raw, ["Mapped Trait(s) (EFO ID)"], required=False),
    "pgs_development_method": _first_existing(pgscatalog_raw, ["PGS Development Method"], required=False),
    "pgs_development_details": _first_existing(pgscatalog_raw, ["PGS Development Details/Relevant Parameters"], required=False),
    "original_genome_build": _first_existing(pgscatalog_raw, ["Original Genome Build"], required=False),
    "number_of_variants": _to_nullable_int(_first_existing(pgscatalog_raw, ["Number of Variants"], required=False)),
    "number_of_interaction_terms": _to_nullable_int(_first_existing(pgscatalog_raw, ["Number of Interaction Terms"], required=False)),
    "type_of_variant_weight": _first_existing(pgscatalog_raw, ["Type of Variant Weight"], required=False),
    "pgp_id": _first_existing(pgscatalog_raw, ["PGS Publication (PGP) ID"], required=False),
    "publication_pmid": _to_nullable_int(_first_existing(pgscatalog_raw, ["Publication (PMID)"], required=False)),
    "publication_doi": _first_existing(pgscatalog_raw, ["Publication (doi)"], required=False),
    "score_and_results_match_original_publication": _to_bool(_first_existing(pgscatalog_raw, ["Score and results match the original publication"], required=False)),
    "ancestry_distribution_source_of_variant_associations_gwas": _first_existing(pgscatalog_raw, ["Ancestry Distribution (%) - Source of Variant Associations (GWAS)"], required=False),
    "ancestry_distribution_score_development_training": _first_existing(pgscatalog_raw, ["Ancestry Distribution (%) - Score Development/Training"], required=False),
    "ancestry_distribution_pgs_evaluation": _first_existing(pgscatalog_raw, ["Ancestry Distribution (%) - PGS Evaluation"], required=False),
    "ftp_link": _first_existing(pgscatalog_raw, ["FTP link"], required=False),
    "release_date": pd.to_datetime(_first_existing(pgscatalog_raw, ["Release Date"], required=False), errors="coerce").dt.date,
    "license_terms_of_use": _first_existing(pgscatalog_raw, ["License/Terms of Use"], required=False),
})

pgs_publications = pd.DataFrame({
    "pgp_id": _first_existing(publications_raw, ["PGS Publication/Study (PGP) ID"]),
    "first_author": _first_existing(publications_raw, ["First Author"], required=False),
    "title": _first_existing(publications_raw, ["Title"], required=False),
    "journal_name": _first_existing(publications_raw, ["Journal Name"], required=False),
    "publication_date": pd.to_datetime(_first_existing(publications_raw, ["Publication Date"], required=False), errors="coerce").dt.date,
    "release_date": pd.to_datetime(_first_existing(publications_raw, ["Release Date"], required=False), errors="coerce").dt.date,
    "authors": _first_existing(publications_raw, ["Authors"], required=False),
    "digital_object_identifier_doi": _first_existing(publications_raw, ["digital object identifier (doi)"], required=False),
    "pubmed_id_pmid": _first_existing(publications_raw, ["PubMed ID (PMID)"], required=False),
})

ontology_mappings = pd.DataFrame({
    "ontology_id": _first_existing(ontology_raw, ["Ontology Trait ID"]),
    "ontology_label": _first_existing(ontology_raw, ["Ontology Trait Label"], required=False),
    "ontology_description": _first_existing(ontology_raw, ["Ontology Trait Description"], required=False),
    "ontology_url": _first_existing(ontology_raw, ["Ontology URL"], required=False),
})

hazard_point, hazard_ci_lower, hazard_ci_upper = _parse_effect_with_ci(
    _first_existing(performance_raw, ["Hazard Ratio (HR)"], required=False)
)
odds_point, odds_ci_lower, odds_ci_upper = _parse_effect_with_ci(
    _first_existing(performance_raw, ["Odds Ratio (OR)"], required=False)
)

pgs_performance = pd.DataFrame({
    "ppm_id": _first_existing(performance_raw, ["PGS Performance Metric (PPM) ID"]),
    "pgs_id": _first_existing(performance_raw, ["Evaluated Score"]),
    "pss_id": _first_existing(performance_raw, ["PGS Sample Set (PSS)"], required=False),
    "pgp_id": _first_existing(performance_raw, ["PGS Publication (PGP) ID"], required=False),
    "reported_trait": _first_existing(performance_raw, ["Reported Trait"], required=False),
    "covariates_included_in_model": _first_existing(performance_raw, ["Covariates Included in the Model"], required=False),
    "pgs_performance_other_relevant_info": _first_existing(performance_raw, ["PGS Performance: Other Relevant Information"], required=False),
    "publication_pmid": _to_nullable_int(_first_existing(performance_raw, ["Publication (PMID)"], required=False)),
    "publication_doi": _first_existing(performance_raw, ["Publication (doi)"], required=False),
    "hazard_ratio": hazard_point,
    "hazard_ratio_ci_lower": hazard_ci_lower,
    "hazard_ratio_ci_upper": hazard_ci_upper,
    "odds_ratio": odds_point,
    "odds_ratio_ci_lower": odds_ci_lower,
    "odds_ratio_ci_upper": odds_ci_upper,
    "beta": _parse_point_estimate(_first_existing(performance_raw, ["Beta"], required=False)),
    "auroc": _parse_point_estimate(_first_existing(performance_raw, ["Area Under the Receiver-Operating Characteristic Curve (AUROC)"], required=False)),
    "concordance_statistic": _parse_point_estimate(_first_existing(performance_raw, ["Concordance Statistic (C-index)"], required=False)),
    "other_metric": _first_existing(performance_raw, ["Other Metric(s)"], required=False),
})

score_development_samples = pd.DataFrame({
    "pgs_id": _first_existing(score_dev_raw, ["Polygenic Score (PGS) ID"]),
    "stage_of_pgs_development": _first_existing(score_dev_raw, ["Stage of PGS Development"], required=False),
    "individuals_development": _to_nullable_int(_first_existing(score_dev_raw, ["Number of Individuals"], required=False)),
    "cases_development": _to_nullable_int(_first_existing(score_dev_raw, ["Number of Cases"], required=False)),
    "controls_development": _to_nullable_int(_first_existing(score_dev_raw, ["Number of Controls"], required=False)),
    "percent_male_development": pd.to_numeric(_first_existing(score_dev_raw, ["Percent of Participants Who are Male"], required=False), errors="coerce"),
    "broad_ancestry_category": _first_existing(score_dev_raw, ["Broad Ancestry Category", "Broad Ancestry Category(ies)", "Broad ancestry category"], required=False),
})

evaluation_sample_sets = pd.DataFrame({
    "pss_id": _first_existing(eval_sets_raw, ["PGS Sample Set (PSS)"]),
    "pgs_id": _first_existing(eval_sets_raw, ["Polygenic Score (PGS) ID", "Evaluated Score", "PGS ID"]),
    "individuals_evaluation": _to_nullable_int(_first_existing(eval_sets_raw, ["Number of Individuals"], required=False)),
    "cases_evaluation": _to_nullable_int(_first_existing(eval_sets_raw, ["Number of Cases"], required=False)),
    "controls_evaluation": _to_nullable_int(_first_existing(eval_sets_raw, ["Number of Controls"], required=False)),
    "percent_male_evaluation": pd.to_numeric(_first_existing(eval_sets_raw, ["Percent of Participants Who are Male"], required=False), errors="coerce"),
    "broad_ancestry_category": _first_existing(eval_sets_raw, ["Broad Ancestry Category", "Broad Ancestry Category(ies)", "Broad ancestry category"], required=False),
    "country_of_recruitment": _first_existing(eval_sets_raw, ["Country of Recruitment", "Country/Countries of Recruitment", "Countries of recruitment"], required=False),
    "cohort": _first_existing(
        eval_sets_raw,
        [
            "Cohort",
            "Cohort(s)",
            "Cohorts",
            "Cohort(s) Included",
            "Cohort Name",
        ],
        required=False,
    ),
})

# Expand many-to-many style PSS->PGS links into one row per pgs_id.
evaluation_sample_sets["pgs_id"] = evaluation_sample_sets["pgs_id"].apply(_split_pgs_ids)
evaluation_sample_sets = evaluation_sample_sets.explode("pgs_id", ignore_index=True)

# Remove blank keys so inserts do not fail on PK-only empty strings.
for dataframe, pk_col in [
    (pgscatalog_data, "pgs_id"),
    (pgs_publications, "pgp_id"),
    (ontology_mappings, "ontology_id"),
    (pgs_performance, "ppm_id"),
    (score_development_samples, "pgs_id"),
    (evaluation_sample_sets, "pss_id"),
]:
    dataframe[pk_col] = dataframe[pk_col].astype(str).str.strip()
    dataframe.drop(dataframe[dataframe[pk_col].isin(["", "nan", "None"])].index, inplace=True)

# Clean exploded pgs_id values and drop rows that still have no usable pgs_id.
evaluation_sample_sets["pgs_id"] = evaluation_sample_sets["pgs_id"].astype(str).str.strip()
evaluation_sample_sets.drop(
    evaluation_sample_sets[evaluation_sample_sets["pgs_id"].isin(["", "nan", "None"])].index,
    inplace=True,
)

evaluation_sample_sets = evaluation_sample_sets.drop_duplicates(
    subset=["pss_id", "pgs_id", "individuals_evaluation", "cases_evaluation", "controls_evaluation", "percent_male_evaluation", "broad_ancestry_category", "country_of_recruitment", "cohort"]
)

pgscatalog_data = _clean_df_for_sql(pgscatalog_data)
pgs_publications = _clean_df_for_sql(pgs_publications)
ontology_mappings = _clean_df_for_sql(ontology_mappings)
pgs_performance = _clean_df_for_sql(pgs_performance)
score_development_samples = _clean_df_for_sql(score_development_samples)
evaluation_sample_sets = _clean_df_for_sql(evaluation_sample_sets)


# -----------------------------
# 3) Connect and load into DB
# -----------------------------
DB_USER = os.getenv("DB_USER", os.getenv("POSTGRES_USER", "postgres"))
DB_PASSWORD = os.getenv("DB_PASSWORD", os.getenv("POSTGRES_PASSWORD", "zod50902"))
DB_NAME = os.getenv("DB_NAME", os.getenv("POSTGRES_DB", "snpster_db"))
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = int(os.getenv("DB_PORT", "5433"))

conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
)

try:
    with conn.cursor() as cur:
        # Compatibility for older live schemas still using legacy column names.
        eval_cols = set(_get_table_columns(cur, "data_libraries", "evaluation_sample_sets"))
        if "pgs_id" not in eval_cols and "evaluated_score" in eval_cols and "pgs_id" in evaluation_sample_sets.columns:
            evaluation_sample_sets = evaluation_sample_sets.rename(columns={"pgs_id": "evaluated_score"})
            print("Mapped evaluation_sample_sets.pgs_id -> evaluated_score for live DB compatibility")

        perf_cols = set(_get_table_columns(cur, "data_libraries", "pgs_performance"))
        if "ppm_id" not in perf_cols and "performance_id" in perf_cols and "ppm_id" in pgs_performance.columns:
            pgs_performance = pgs_performance.rename(columns={"ppm_id": "performance_id"})
            print("Mapped pgs_performance.ppm_id -> performance_id for live DB compatibility")

        # Truncate all target metadata tables in one statement.
        cur.execute(
            """
            TRUNCATE TABLE
                data_libraries.pgs_performance,
                data_libraries.evaluation_sample_sets,
                data_libraries.score_development_samples,
                data_libraries.pgs_publications,
                data_libraries.ontology_mappings,
                data_libraries.pgscatalog_data
            RESTART IDENTITY CASCADE
            """
        )

        _bulk_insert(cur, "data_libraries", "pgscatalog_data", pgscatalog_data)
        _bulk_insert(cur, "data_libraries", "pgs_publications", pgs_publications)
        _bulk_insert(cur, "data_libraries", "ontology_mappings", ontology_mappings)
        _bulk_insert(cur, "data_libraries", "score_development_samples", score_development_samples)
        _bulk_insert(cur, "data_libraries", "evaluation_sample_sets", evaluation_sample_sets)
        _bulk_insert(cur, "data_libraries", "pgs_performance", pgs_performance)

    conn.commit()
    print("Metadata refresh complete.")
except Exception:
    conn.rollback()
    raise
finally:
    conn.close()

Inserted 6972 rows into data_libraries.pgscatalog_data
Inserted 823 rows into data_libraries.pgs_publications
Inserted 807 rows into data_libraries.ontology_mappings
Inserted 15633 rows into data_libraries.score_development_samples
Inserted 20281 rows into data_libraries.evaluation_sample_sets
Inserted 22925 rows into data_libraries.pgs_performance
Metadata refresh complete.
